# NumPy 계산 경로 해부기: 행렬곱과 브로드캐스팅

이 노트북은 행렬곱과 브로드캐스팅을 단순 shape 확인이 아니라 계산 경로 해부 방식으로 다룬다.

- 행렬곱: 결과 칸 `C[i,j]`가 `A`의 `i`번 행과 `B`의 `j`번 열에서 어떻게 만들어지는지 확인한다.
- 브로드캐스팅: 결과 칸 `C[i,j]`가 원본 `A`, `B`의 어느 칸을 참조하는지 확인한다.
- 신경망 연결: `X @ W + b`에서 편향 `b`가 모든 샘플에 어떻게 더해지는지 확인한다.


In [ ]:
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])
B = np.array([[7, 8],
              [9, 10],
              [11, 12]])

C = A @ B
print('A.shape =', A.shape)
print('B.shape =', B.shape)
print('C.shape =', C.shape)
print(C)

# C[0,1] 직접 계산
terms = [A[0, k] * B[k, 1] for k in range(A.shape[1])]
print('C[0,1] 계산 항:', terms)
print('C[0,1] =', sum(terms))

In [ ]:
def matmul_by_hand(A, B):
    m, r = A.shape
    r2, n = B.shape
    assert r == r2, '가운데 차원이 같아야 한다.'
    C = np.zeros((m, n), dtype=A.dtype)
    trace = {}
    for i in range(m):
        for j in range(n):
            terms = []
            for k in range(r):
                value = A[i, k] * B[k, j]
                terms.append((i, k, k, j, value))
                C[i, j] += value
            trace[(i, j)] = terms
    return C, trace

C2, trace = matmul_by_hand(A, B)
print(C2)
print('C[0,1] trace:')
for i, k1, k2, j, value in trace[(0, 1)]:
    print(f'A[{i},{k1}] * B[{k2},{j}] = {A[i,k1]} * {B[k2,j]} = {value}')

## 브로드캐스팅 인덱스 해부

브로드캐스팅은 결과 shape만 보는 것이 아니라, 결과 칸이 원본 배열의 어느 인덱스를 읽는지까지 확인해야 한다.


In [ ]:
def broadcast_result_shape(shape_a, shape_b):
    L = max(len(shape_a), len(shape_b))
    a = (1,) * (L - len(shape_a)) + tuple(shape_a)
    b = (1,) * (L - len(shape_b)) + tuple(shape_b)
    out = []
    for da, db in zip(a, b):
        if da == db:
            out.append(da)
        elif da == 1:
            out.append(db)
        elif db == 1:
            out.append(da)
        else:
            raise ValueError(f'브로드캐스팅 실패: {da}와 {db}가 충돌한다.')
    return tuple(out)

def source_index(original_shape, result_index, result_shape):
    original_shape = tuple(original_shape)
    result_shape = tuple(result_shape)
    offset = len(result_shape) - len(original_shape)
    aligned = (1,) * offset + original_shape
    aligned_idx = []
    for size, idx in zip(aligned, result_index):
        aligned_idx.append(0 if size == 1 else idx)
    return tuple(aligned_idx[offset:])

A = np.array([[1, 2, 3],
              [4, 5, 6]])
b = np.array([10, 20, 30])
shape = broadcast_result_shape(A.shape, b.shape)
print('결과 shape:', shape)

result_index = (1, 2)
ia = source_index(A.shape, result_index, shape)
ib = source_index(b.shape, result_index, shape)
print('C[1,2]는 A', ia, '와 b', ib, '에서 온다.')
print('값:', A[ia] + b[ib])

## 인터랙티브 HTML 실행

아래 셀은 같은 폴더에 있는 `numpy_matrix_broadcasting_dissection_lab.html`을 노트북 안에 표시한다. 파일을 단독으로 브라우저에서 열어도 된다.


In [ ]:
from IPython.display import IFrame, display
from pathlib import Path

html_path = Path('numpy_matrix_broadcasting_dissection_lab.html')
if html_path.exists():
    display(IFrame(str(html_path), width='100%', height=900))
else:
    print('HTML 파일을 같은 폴더에 두고 다시 실행하라:', html_path)